# 🖼️ Day 04a: CNN — Convolutional Neural Networks

---

## 🎯 What You'll Master Today
- Conv2d, MaxPool2d, padding, stride
- Feature map size calculation
- Build a CNN classifier
- CNN architecture patterns

**Goal:** Build a CNN from memory. Know how to calculate output dimensions.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  CNN — Convolution Intuition                                     ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Input image: (batch, channels, height, width)                  ║
║               (B, C, H, W)                                       ║
║                                                                   ║
║  Conv2d(in_ch, out_ch, kernel_size):                            ║
║    • Slides kernel across image                                  ║
║    • Each kernel detects one feature (edge, texture, etc.)      ║
║    • out_ch = number of feature detectors                       ║
║                                                                   ║
║  Output size formula:                                            ║
║    out = (in + 2·padding - kernel_size) / stride + 1            ║
║                                                                   ║
║  Typical pattern:                                                ║
║    Conv → BN → ReLU → Pool → ... → Flatten → FC → Output      ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# 1. Conv2d Basics — Understanding Dimensions
# ============================================================

# Input: 1 grayscale image, 28x28 (like MNIST)
x = torch.randn(1, 1, 28, 28)  # (batch, channels, H, W)
print(f"Input: {x.shape}")

# Conv2d(in_channels, out_channels, kernel_size)
conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)  # same padding
out1 = conv1(x)
# out = (28 + 2*1 - 3) / 1 + 1 = 28 (same size with padding=1)
print(f"Conv2d(1→16, k=3, p=1): {out1.shape}")

conv2 = nn.Conv2d(1, 16, kernel_size=3, padding=0)  # no padding
out2 = conv2(x)
# out = (28 + 0 - 3) / 1 + 1 = 26
print(f"Conv2d(1→16, k=3, p=0): {out2.shape}")

conv3 = nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1)
out3 = conv3(x)
# out = (28 + 2 - 3) / 2 + 1 = 14
print(f"Conv2d(1→16, k=3, s=2, p=1): {out3.shape}")

# MaxPool2d
pool = nn.MaxPool2d(kernel_size=2, stride=2)
out4 = pool(out1)
print(f"MaxPool2d(k=2, s=2): {out1.shape} → {out4.shape}")

# Formula reminder
print("\n💡 Output size = (input + 2*padding - kernel) / stride + 1")
print("   padding='same' equivalent: padding = kernel_size // 2")

In [ ]:
# ============================================================
# 2. Parameter Count in Conv Layers
# ============================================================

# Conv2d params = out_ch * (in_ch * kernel_h * kernel_w + 1)  (+1 for bias)
conv = nn.Conv2d(3, 16, kernel_size=3)  # RGB input, 16 filters
params = sum(p.numel() for p in conv.parameters())
expected = 16 * (3 * 3 * 3) + 16  # 16 filters × (3 channels × 3×3 kernel) + 16 biases

print(f"Conv2d(3→16, k=3):")
print(f"  Weight shape: {conv.weight.shape}")  # (out_ch, in_ch, kH, kW)
print(f"  Bias shape: {conv.bias.shape}")
print(f"  Total params: {params} (expected: {expected})")

# Compare with FC layer for same input
# A 28×28×3 input with FC → 16 neurons
fc = nn.Linear(28*28*3, 16)
fc_params = sum(p.numel() for p in fc.parameters())
print(f"\nLinear(28*28*3→16): {fc_params} params")
print(f"Conv2d(3→16, k=3):  {params} params")
print(f"\n💡 Conv: {params} vs FC: {fc_params} — CNNs are parameter efficient!")

In [ ]:
# ============================================================
# 3. CNN Classifier — MNIST-style
# ============================================================

class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Feature extractor
        self.features = nn.Sequential(
            # Block 1: 1×28×28 → 32×14×14
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # → 32×28×28
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # → 32×14×14
            
            # Block 2: 32×14×14 → 64×7×7
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # → 64×14×14
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # → 64×7×7
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),                    # 64×7×7 = 3136
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),      # logits
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CNN(num_classes=10)

# Trace shapes through the network
x = torch.randn(2, 1, 28, 28)
print("Shape trace:")
print(f"  Input:        {x.shape}")

# Manual trace
out = model.features[0](x)  # Conv
print(f"  After Conv1:  {out.shape}")
out = model.features[:4](x)  # Conv+BN+ReLU+Pool
print(f"  After Block1: {out.shape}")
out = model.features(x)  # Full features
print(f"  After Block2: {out.shape}")
out = model(x)  # Full model
print(f"  Output:       {out.shape}")

total = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total:,}")

In [ ]:
# ============================================================
# 4. CNN for RGB Images (CIFAR-style)
# ============================================================

class CIFARCNN(nn.Module):
    """For 3×32×32 RGB images, 10 classes."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # 3×32×32 → 32×16×16
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            # 32×16×16 → 64×8×8
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            # 64×8×8 → 128×4×4
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),  # → 128×1×1 (any input size!)
            nn.Flatten(),
            nn.Linear(128, 10),
        )
    
    def forward(self, x):
        return self.classifier(self.features(x))

model_cifar = CIFARCNN()
x = torch.randn(4, 3, 32, 32)
out = model_cifar(x)
print(f"CIFAR CNN: {x.shape} → {out.shape}")
print(f"Params: {sum(p.numel() for p in model_cifar.parameters()):,}")

print("\n💡 AdaptiveAvgPool2d(1) → output is always 1×1, regardless of input size!")
print("   This replaces the need to calculate exact FC input dimensions.")

In [ ]:
# ============================================================
# 5. CNN Architecture Patterns
# ============================================================

print("""
CNN ARCHITECTURE PATTERNS
═════════════════════════

Classic Pattern:
  [Conv → BN → ReLU → Pool] × N → Flatten → FC → Output

VGG-style (3×3 only):
  [Conv3×3 → Conv3×3 → Pool] × N  (deeper but simple)

ResNet-style (skip connections):
  x → [Conv → BN → ReLU → Conv → BN] + x → ReLU

Common Hyperparameters:
  ┌──────────────┬────────────────────────────────┐
  │ Layer        │ Typical Values                 │
  ├──────────────┼────────────────────────────────┤
  │ Conv kernel  │ 3×3 (most common), 1×1 (proj.) │
  │ Padding      │ same (kernel//2), valid (0)     │
  │ Stride       │ 1 (keep size), 2 (downsample)  │
  │ Pool         │ MaxPool 2×2, AvgPool, Adaptive │
  │ Channels     │ 32→64→128→256 (double pattern) │
  │ FC dropout   │ 0.5 (heavy), 0.25 (light)      │
  └──────────────┴────────────────────────────────┘

  💡 "Double channels, halve spatial" = the universal CNN pattern
""")

---

## 🗺️ CNN Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  CNN ESSENTIALS                                                  ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Conv2d(in_ch, out_ch, kernel, stride, padding)                 ║
║  Output = (input + 2p - k) / s + 1                              ║
║                                                                   ║
║  Building block: Conv → BN → ReLU → Pool                       ║
║  Pattern: double channels, halve spatial at each block          ║
║                                                                   ║
║  Efficiency: Conv shares weights → fewer params than FC         ║
║  AdaptiveAvgPool2d(1) → size-agnostic before FC                ║
║                                                                   ║
║  Famous architectures:                                           ║
║    LeNet → AlexNet → VGG → GoogLeNet → ResNet → EfficientNet  ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Why CNN over FC for images? | Weight sharing → fewer params. Translation invariance. Preserves spatial structure |
| 2 | How to calculate output size? | (input + 2*padding - kernel) / stride + 1 |
| 3 | What do early vs late conv layers learn? | Early: edges, textures. Late: parts, objects. Hierarchical feature learning |
| 4 | What is a 1×1 convolution? | Channel-wise projection. Reduces/increases channels without changing spatial dims. Used in Inception, ResNet |
| 5 | BatchNorm before or after ReLU? | Debated. Original paper: BN before ReLU. Modern practice: either works. Be consistent |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Input shape: (B, C, H, W) — channels first!         │
## │  • Output = (in + 2p - k) / s + 1                      │
## │  • Block: Conv → BN → ReLU → Pool                      │
## │  • Pattern: double channels, halve spatial               │
## │  • AdaptiveAvgPool2d(1) before FC = size agnostic       │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **RNN & LSTM** — sequence modeling